In [1]:
import warnings
warnings.filterwarnings("ignore", message="Attempting to use hipBLASLt")

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

### Подготовка наборов данных.

В этом блокноте посмотрим на обучающий и тестовый наборы данных. 

Выполним очистку данных (если нужно) и сохраним в .csv файлы для быстрой загрузки в датафреймы в последующем.

In [3]:
# Читаем данные для обучения
df_data = pd.read_json(path_or_buf="./data/data_for_train.jsonl", lines=True)

In [4]:
# Убеждаемся, что две таргетные колонки полностью совпадают
(df_data['relevance'] == df_data['relevance_new']).value_counts()

True    34094
Name: count, dtype: int64

In [5]:
# Оставляем одну таргетную колонку (последнюю)
df_data = df_data.drop(columns=['relevance'])
df_data = df_data.rename(columns={'relevance_new': 'relevance'})

# На всякий случай удаляем полные дубликаты (если бы они были)
df_data = df_data.drop_duplicates()

# Смотрим количество строк и наличие пустых данных NaN
print(df_data.shape)
print(df_data.isna().sum())

# Cохраняем в .csv файл
df_data.to_csv('./data/df_data.csv', index=False)

(34094, 8)
Text                                  0
address                               0
name                                  0
normalized_main_rubric_name_ru        0
permalink                             0
prices_summarized                 14041
reviews_summarized                 1499
relevance                             0
dtype: int64


In [6]:
# Посмотрим глазами первые 5 записей
df_data.head()

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,reviews_summarized,relevance
0,Стриптиз клубы ижевск,"Удмуртская Республика, Ижевск, Береговая улица",More,Яхт-клуб,184649161205,"Яхт-клуб More предлагает прогулки на яхтах, ар...",Общий обзор отзывов: яхт-клуб «More» в Ижевске...,0.0
1,Подшипник KAMAZ 45105000000000,"Ростовская область, Таганрог, улица Ломоносова...","Подшипник; Kompaniya Podshipnik; Подшипник, ТД...",Магазин автозапчастей и автотоваров,84348429597,NaN,Организация занимается продажей автозапчастей ...,0.1
2,такелажные компании,"Киров, улица Менделеева, 2",Кировская такелажная компания; Kirovskaya Take...,Строительные леса,1095265306,Кировская такелажная компания предлагает строп...,Организация занимается продажей строительных л...,1.0
3,баннеры,"Свердловская область, Екатеринбург, улица Карл...","Дабл, Онлайн-Полиграфия; Double Print; Студия ...",Полиграфические услуги,1221953140,NaN,Организация занимается полиграфическими услуга...,1.0
4,лучший ресторан москвы 2016,"Москва, улица Крымский Вал, 9с1",Сыроварня; Syrovarnya; Сыроварня в Парке Горьк...,Ресторан,191911335353,Ресторан предлагает разнообразные блюда: от за...,Организация занимается приготовлением и подаче...,1.0


In [7]:
# Посмотрим делальнее на одну произвольную запись
idx = 0
print(df_data.iloc[idx]['Text'], '\n')
print(df_data.iloc[idx]['address'], '\n')
print(df_data.iloc[idx]['name'], '\n')
print(df_data.iloc[idx]['normalized_main_rubric_name_ru'], '\n')
print(df_data.iloc[idx]['prices_summarized'], '\n')
print(df_data.iloc[idx]['reviews_summarized'])

Стриптиз клубы ижевск 

Удмуртская Республика, Ижевск, Береговая улица 

More 

Яхт-клуб 

Яхт-клуб More предлагает прогулки на яхтах, аренду с инструктором или без, а также программу для знакомства с яхтингом | прогулка на яхте | аренда яхты с инструктором | аренда яхты без шкипера | программа «Знакомство с яхтингом» 

Общий обзор отзывов: яхт-клуб «More» в Ижевске предоставляет услуги по обучению и организации мероприятий на воде. Отзывы исключительно положительные: хвалят атмосферу, профессионализм персонала, качество услуг и удобное расположение. | 1. Хвалят эмоции и кайф от посещения | 2. Отмечают хорошее место | 3. Подчёркивают отличную локацию | 4. Считают место топовым в Ижевске | 5. Рекомендуют для любителей парусного спорта | 6. Отмечают крутость места | 7. Описывают как место силы и отличного настроения | 8. Подчёркивают душевный подход к организации гонок | 9. Советуют посетить клуб | 10. Рекомендуют место | 11. Отмечают красоту мест и отдых | 12. Подчёркивают создание атмо

In [8]:
# Делим df_data на train/val и сохраняем в .csv файлы

SEED = 42
VAL_SIZE = 0.1

# Делим по запросам, а не по строкам, чтобы один запрос не попал одновременно в train и val.
unique_queries = df_data["Text"].unique()
unique_queries = np.array(unique_queries.tolist())

train_queries, val_queries = train_test_split(
    unique_queries,
    test_size=VAL_SIZE,
    random_state=SEED,
)

df_train = df_data[df_data["Text"].isin(train_queries)].copy()
df_val   = df_data[df_data["Text"].isin(val_queries)].copy()

print(f"Train: {len(df_train)} строк, {len(train_queries)} уникальных запросов")
print(f"Val:   {len(df_val)} строк, {len(val_queries)} уникальных запросов")

print("\n--- Распределение классов в df_train ---")
print(df_train["relevance"].value_counts(normalize=True).round(3))

print("\n--- Распределение классов в df_val ---")
print(df_val["relevance"].value_counts(normalize=True).round(3))

df_train.to_csv('./data/df_train.csv', index=False)
df_val.to_csv('./data/df_val.csv', index=False)

Train: 30692 строк, 8325 уникальных запросов
Val:   3402 строк, 926 уникальных запросов

--- Распределение классов в df_train ---
relevance
1.0    0.452
0.0    0.416
0.1    0.132
Name: proportion, dtype: float64

--- Распределение классов в df_val ---
relevance
1.0    0.466
0.0    0.387
0.1    0.147
Name: proportion, dtype: float64


In [9]:
# Делаем дополнительный набор: удаляем строки с значением таргетной переменной 0.1 
df_data_drop_01 = df_data[df_data['relevance'] != 0.1]

# Сохраняем в .csv файл
df_data_drop_01.to_csv('./data/df_data_drop_01.csv', index=False)

In [10]:
# Делим df_data_drop_01 на train/val

# Делим по запросам, а не по строкам, чтобы один запрос не попал одновременно в train и val.
unique_queries = df_data_drop_01["Text"].unique()
unique_queries = np.array(unique_queries.tolist())

train_queries, val_queries = train_test_split(
    unique_queries,
    test_size=VAL_SIZE,
    random_state=SEED,
)

df_train_drop_01 = df_data_drop_01[df_data_drop_01["Text"].isin(train_queries)].copy()
df_val_drop_01   = df_data_drop_01[df_data_drop_01["Text"].isin(val_queries)].copy()

print(f"Train: {len(df_train_drop_01)} строк, {len(train_queries)} уникальных запросов")
print(f"Val:   {len(df_val_drop_01)} строк, {len(val_queries)} уникальных запросов")

print("\n--- Распределение классов в df_train_drop_01 ---")
print(df_train_drop_01["relevance"].value_counts(normalize=True).round(3))

print("\n--- Распределение классов в df_val_drop_01 ---")
print(df_val_drop_01["relevance"].value_counts(normalize=True).round(3))

#  Сохраняем в .csv файлы
df_train_drop_01.to_csv('./data/df_train_drop_01.csv', index=False)
df_val_drop_01.to_csv('./data/df_val_drop_01.csv', index=False)

Train: 26585 строк, 7992 уникальных запросов
Val:   2945 строк, 889 уникальных запросов

--- Распределение классов в df_train_drop_01 ---
relevance
1.0    0.527
0.0    0.473
Name: proportion, dtype: float64

--- Распределение классов в df_val_drop_01 ---
relevance
0.0    0.505
1.0    0.495
Name: proportion, dtype: float64


In [11]:
# Читаем данные для тестирования
df_test = pd.read_json(path_or_buf="./data/data_for_eval.jsonl", lines=True)

In [12]:
# Переименовываем таргетную колонку для единообразия
df_test = df_test.rename(columns={'relevance_new': 'relevance'})

# Смотрим количество строк и наличие пустых данных NaN
print(df_test.shape)
print(df_test.isna().sum())

# Cохраняем в .csv файл
df_test.to_csv('./data/df_test.csv', index=False)

(570, 8)
Text                                0
address                             0
name                                0
normalized_main_rubric_name_ru      0
permalink                           0
prices_summarized                 236
reviews_summarized                 21
relevance                           0
dtype: int64


In [13]:
# Посмотрим глазами первые 5 записей
df_test.head()

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,reviews_summarized,relevance
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,NaN,"Организация занимается продажей табака, курите...",1.0
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,"Организация PioNero — это кафе, бар и ресторан...",0.0
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...","Организация занимается стоматологическими, кос...",1.0
3,спортзал где 1 занятие бесплатно,"Москва, Страстной бульвар, 13А",Унца Унца Спорт; Unza Unza Sport,Фитнес-клуб,201938477844,Фитнес-клуб предлагает пробные занятия по разл...,Организация «Унца Унца Спорт» предоставляет ус...,0.1
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,Организация занимается продажей бытовой техник...,1.0


In [14]:
print("--- Распределение классов в df_test ---")
print(df_test["relevance"].value_counts(normalize=True).round(3))

--- Распределение классов в df_test ---
relevance
1.0    0.556
0.0    0.321
0.1    0.123
Name: proportion, dtype: float64


In [15]:
# Делаем дополнительный набор: удаляем строки с значением таргетной переменной 0.1
df_test_drop_01 = df_test[df_test['relevance'] != 0.1]

# Смотрим количество строк и наличие пустых данных NaN
print(df_test_drop_01.shape)
print(df_test_drop_01.isna().sum())

# Сохраняем в .csv файл
df_test_drop_01.to_csv('./data/df_test_drop_01.csv', index=False)

(500, 8)
Text                                0
address                             0
name                                0
normalized_main_rubric_name_ru      0
permalink                           0
prices_summarized                 207
reviews_summarized                 15
relevance                           0
dtype: int64


In [16]:
print("--- Распределение классов в df_test_drop_01 ---")
print(df_test_drop_01["relevance"].value_counts(normalize=True).round(3))

--- Распределение классов в df_test_drop_01 ---
relevance
1.0    0.634
0.0    0.366
Name: proportion, dtype: float64


#### ИТОГО
Т.о. мы подготовили два набора данных (с значениями 0.1 и без них) для обучения/валидации:
- df_train, df_val (+ полный df_data)
- df_train_drop_01, df_val_drop_01 (+ полный df_data_drop_01)

И два набора данных для тестирования (с значениями 0.1 и без них):
- df_test
- df_test_drop_01 

##### *В последствии использовались только наборы "..._drop_01" (без значений 0.1) - давали лучший результат.*